# Day 9: Machine Learning Fundamentals - Regression, Feature Selection, and Classification

## Overview
This Jupyter Notebook contains detailed course notes and code demonstrations based on whiteboard discussions covering:
1. Data Splitting Techniques (Train vs. Test)
2. Multiple Linear Regression Models & Mathematical Foundations
3. Feature Selection & Multicollinearity Analysis
4. Practical Code Implementation in Python (`scikit-learn`, `pandas`, `numpy`)
5. Intro to Supervised Classification


## 1. Train-Test Split Strategy

To evaluate how well a machine learning model generalizes to unseen data, we divide our dataset into two distinct subsets:
* **Training Data (~80%):** Used by the algorithm to learn underlying patterns and fit model parameters (e.g., weights $b_1, b_2, \dots$).
* **Testing Data (~20%):** Reserved strictly for evaluation. Performance metrics (such as $R^2$ or RMSE) calculated on this set reflect real-world performance.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Generate a synthetic dataset representing house features and prices
np.random.seed(42)
n_samples = 100

data = {
    'Sinks': np.random.randint(1, 6, size=n_samples),
    'SqFt': np.random.randint(500, 3500, size=n_samples),
    'Baths': np.random.randint(1, 4, size=n_samples),
    'Beds': np.random.randint(1, 5, size=n_samples)
}

df = pd.DataFrame(data)
# Price formula: 50k base + 15k*Sinks + 250*SqFt + 20k*Baths + 10k*Beds + noise
df['Price'] = 50000 + (15000 * df['Sinks']) + (250 * df['SqFt']) + (20000 * df['Baths']) + (10000 * df['Beds']) + np.random.normal(0, 15000, n_samples)

# Separate predictor features (X) and target variable (y)
X = df[['Sinks', 'SqFt', 'Baths', 'Beds']]
y = df['Price']

# Perform an 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Total dataset size: {len(df)} samples")
print(f"Training set size: {len(X_train)} samples (80%)")
print(f"Testing set size: {len(X_test)} samples (20%)")


## 2. Multiple Linear Regression Model & Equations

Linear regression models the relationship between a continuous target variable $Y$ and one or more predictor features $X_1, X_2, \dots, X_n$.

### Mathematical Formulation
$$\hat{Y} = b_0 + b_1 X_1 + b_2 X_2 + b_3 X_3 + \dots + b_n X_n$$

Where:
* $\hat{Y}$: Predicted target value (e.g., house price).
* $b_0$: Intercept (the baseline target value when all input features equal zero).
* $b_1, b_2, \dots, b_n$: Regression coefficients representing the marginal impact of each feature $X_i$ on target $Y$.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept (b0): ${model.intercept_:,.2f}")
for col, coef in zip(X.columns, model.coef_):
    print(f"Coefficient for {col} (b_i): ${coef:,.2f}")

# Evaluate performance on test set
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nModel R^2 Score on Test Data: {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")


## 3. Feature Selection & Correlation Analysis

### Feature Selection Guidelines
1. **High Correlation with Target ($Y$):** Retain features that demonstrate strong predictive power toward $Y$.
2. **Low Inter-Feature Correlation:** Avoid selecting features ($X_i, X_j$) that share high pairwise correlation (multicollinearity).

### Correlation Matrix Example

| Feature | $X_1$ | $X_2$ | $X_3$ | $Y$ |
| :--- | :--- | :--- | :--- | :--- |
| **$X_1$** | 1.00 | 0.20 | **0.70** | **0.70** |
| **$X_2$** | 0.20 | 1.00 | 0.30 | 0.10 |
| **$X_3$** | **0.70** | 0.30 | 1.00 | **0.65** |
| **$Y$** | **0.70** | 0.10 | **0.65** | 1.00 |

* **Key Takeaway:** $X_1$ and $X_3$ both correlate strongly with $Y$ ($0.70$ and $0.65$). However, because $X_1$ and $X_3$ are highly correlated with each other ($0.70$), including both introduces collinearity redundancy. We prefer picking $X_1$ over $X_3$.


In [ ]:
# Compute correlation matrix
corr_matrix = df.corr()
print('Correlation Matrix:')
print(corr_matrix.round(2))

print('\nFeature Correlations with Price:')
print(corr_matrix['Price'].drop('Price').round(2))


## 4. Supervised Classification Overview

While **Regression** models continuous variables, **Classification** assigns data points into discrete class categories (e.g., Yes/No, High/Low, Class A/B/C).

| Property | Regression | Classification |
| :--- | :--- | :--- |
| **Target Variable** | Continuous scalar value | Categorical class label |
| **Example Tasks** | Predicting price, stock value, temperature | Fraud detection, image labeling, spam filtering |


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Convert continuous price into binary classification target (1 = Above Median, 0 = Below Median)
median_price = df['Price'].median()
df['Price_Category'] = (df['Price'] > median_price).astype(int)

X_cls = df[['Sinks', 'SqFt', 'Baths', 'Beds']]
y_cls = df['Price_Category']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.20, random_state=42)

# Fit Logistic Regression Classifier
clf = LogisticRegression()
clf.fit(X_train_c, y_train_c)

y_pred_c = clf.predict(X_test_c)
acc = accuracy_score(y_test_c, y_pred_c)

print(f"Target Category: Price > ${median_price:,.2f}")
print(f"Logistic Regression Classification Accuracy: {acc * 100:.2f}%")
